# Tien xu ly Celeb-DF v2 -- Cache rPPG/CAN (100 frames)

**Muc tieu:** trich khung hinh + cat khuon mat cho toan bo **518 video trong
`List_of_testing_videos.txt`** voi **100 frames** (thay vi 32 frames o cache Nhom A), dung cho
nhanh rPPG (DeepPhysCAN_v2) va model chinh CAN (DF-FusionModel).

Pipeline tien xu ly **GIONG HET** notebook `prepare-data-celebdf-v2.ipynb` (Nhom A):
cung MTCNN, `FACE_MARGIN=1.3`, `IMG_SIZE=256`, `JPEG_QUALITY=92` -- chi khac `NUM_FRAMES=100`
de rPPG branch co du temporal context (CLIP_LEN_EVAL=96, can >= 97 frames lien tiep).

## Truoc khi chay
1. Add Input -> dataset `reubensuju/celeb-df-v2` (hoac dataset Celeb-DF v2 khac -- sua `CELEBDF_ROOT`).
2. Settings -> Accelerator -> GPU (can cho MTCNN).
3. Settings -> Internet -> ON (tai pretrained MTCNN neu chua cache).

## 1. Cau hinh

In [ ]:
!pip install -q facenet-pytorch --no-deps

In [ ]:
import os, json, cv2, glob, time, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import warnings
warnings.filterwarnings("ignore")

# ==================== DUONG DAN DATASET GOC ====================
CELEBDF_ROOT  = "/kaggle/input/celeb-df-v2"   # SUA neu ten dataset/duong dan khac
LIST_TEST_TXT = os.path.join(CELEBDF_ROOT, "List_of_testing_videos.txt")

assert os.path.isdir(CELEBDF_ROOT), f"Khong tim thay CELEBDF_ROOT: {CELEBDF_ROOT}"
assert os.path.exists(LIST_TEST_TXT), (
    f"Khong tim thay {LIST_TEST_TXT} -- kiem tra lai CELEBDF_ROOT co dung version chua "
    f"List_of_testing_videos.txt hay khong."
)

# ==================== NOI LUU CACHE ANH DA CROP ====================
OUT_DIR = "/kaggle/working/celebdf_v2_rppg_cache"
os.makedirs(OUT_DIR, exist_ok=True)

# ==================== THAM SO TRICH XUAT ====================
# GIONG HET prepare-data-celebdf-v2.ipynb (Nhom A), CHI KHAC NUM_FRAMES=100
NUM_FRAMES   = 100     # Nhom A=32, Nhom B(rPPG/CAN)=100
FACE_MARGIN  = 1.3
IMG_SIZE     = 256
JPEG_QUALITY = 92
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("CELEBDF_ROOT ton tai:", os.path.isdir(CELEBDF_ROOT))
print("OUT_DIR:", OUT_DIR)
print(f"NUM_FRAMES={NUM_FRAMES} (100 frames cho rPPG/CAN)")
print(os.listdir(CELEBDF_ROOT))

def print_disk_usage():
    total, used, free = shutil.disk_usage(OUT_DIR)
    print(f"Disk: {used/1e9:.2f} / {total/1e9:.2f} GB used, {free/1e9:.2f} GB free")

## 2. Doc `List_of_testing_videos.txt` -- xac thuc cheo nhan qua ten thu muc

Khong tin so 0/1 trong file -- chi dung file nay de lay DANH SACH DUONG DAN 518 video chinh thuc.
Nhan that lay tu ten thu muc cha trong path (`Celeb-real`/`YouTube-real` -> 0, `Celeb-synthesis` -> 1).

In [ ]:
FOLDER_TO_LABEL  = {"Celeb-real": 0, "YouTube-real": 0, "Celeb-synthesis": 1}
FOLDER_TO_METHOD = {"Celeb-real": "original", "YouTube-real": "original",
                    "Celeb-synthesis": "Celeb-synthesis"}

rows = []
n_mismatch = 0
with open(LIST_TEST_TXT, "r") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

print(f"Tong so dong trong List_of_testing_videos.txt: {len(lines)}")

for line in lines:
    parts = line.split()
    assert len(parts) == 2, f"Dong khong dung dinh dang '<label> <path>': {line!r}"
    label_in_file, rel_path = parts
    label_in_file = int(label_in_file)

    folder = rel_path.split("/")[0]
    assert folder in FOLDER_TO_LABEL, f"Thu muc la: {folder}"
    label_true = FOLDER_TO_LABEL[folder]   # nguon su that duy nhat
    if label_in_file != label_true:
        n_mismatch += 1

    video_id = rel_path.replace("/", "__").rsplit(".", 1)[0]
    rows.append({
        "video_id": video_id, "split": "test", "label": label_true,
        "method": FOLDER_TO_METHOD[folder],
        "src_path": os.path.join(CELEBDF_ROOT, rel_path),
        "label_in_file_raw": label_in_file,
    })

manifest = pd.DataFrame(rows)
print(f"\nSo dong nhan-trong-file KHAC nhan-suy-tu-thu-muc: {n_mismatch} / {len(manifest)}")
print(manifest.groupby(["label", "method"])["video_id"].count())

missing = manifest[~manifest["src_path"].apply(os.path.exists)]
print(f"\nSo video KHONG tim thay file tren dia: {len(missing)} / {len(manifest)}")
if len(missing) > 0:
    print(missing[["video_id", "src_path"]].head(10).to_string(index=False))
manifest = manifest[manifest["src_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"\nSo video se xu ly: {len(manifest)}")

## 3. Face detector (MTCNN) & ham trich + cat khuon mat -- copy nguyen xi tu prepareData_16072026_v1.ipynb

In [ ]:
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=IMG_SIZE, margin=0, keep_all=False,
              post_process=False, select_largest=True, device=DEVICE)


def crop_face_frame(frame_rgb):
    """Phat hien khuon mat + cat theo margin; fallback center-crop neu khong detect duoc."""
    h, w = frame_rgb.shape[:2]
    try:
        boxes, probs = mtcnn.detect(frame_rgb)
    except Exception:
        boxes, probs = None, None

    if boxes is not None and len(boxes) > 0:
        idx = int(np.argmax(probs))
        x1, y1, x2, y2 = boxes[idx]
        bw, bh = x2 - x1, y2 - y1
        cx, cy = x1 + bw / 2, y1 + bh / 2
        half = max(bw, bh) * FACE_MARGIN / 2
        x1, y1 = int(max(cx - half, 0)), int(max(cy - half, 0))
        x2, y2 = int(min(cx + half, w)), int(min(cy + half, h))
    else:
        side = min(h, w)
        x1, y1 = (w - side) // 2, (h - side) // 2
        x2, y2 = x1 + side, y1 + side

    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        side = min(h, w)
        x1, y1 = (w - side) // 2, (h - side) // 2
        crop = frame_rgb[y1:y1 + side, x1:x1 + side]
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE))


def extract_clip(video_path, out_dir, num_frames=NUM_FRAMES):
    """Trich num_frames khung hinh cach deu nhau, cat mat, luu JPEG."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0

    idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    idx_set = set(idxs.tolist())
    saved, cur = 0, 0
    while cur <= idxs[-1]:
        ret, frame = cap.read()
        if not ret:
            break
        if cur in idx_set:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face = crop_face_frame(frame_rgb)
            save_path = os.path.join(out_dir, f"frame_{saved:04d}.jpg")
            frame_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
            cv2.imwrite(save_path, frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            saved += 1
        cur += 1
    cap.release()
    return saved

## 4. Vong lap xu ly chinh -- co checkpoint/resume

518 video, trich 100 frames/video (nhieu hon cache Nhom A 32 frames).
Giu resume-safe de phong su co.

In [ ]:
from tqdm import tqdm

CHECKPOINT_CSV = os.path.join(OUT_DIR, "manifest_processed.csv")

def process_all(df, out_dir_root):
    results = []
    for i, row in enumerate(tqdm(df.itertuples(), total=len(df))):
        sub = "real" if row.label == 0 else f"fake_{row.method}"
        out_dir = os.path.join(out_dir_root, row.split, sub, row.video_id)

        if os.path.isdir(out_dir) and len(glob.glob(os.path.join(out_dir, "*.jpg"))) >= NUM_FRAMES:
            results.append(dict(video_id=row.video_id, split=row.split, label=row.label,
                                 method=row.method, out_dir=out_dir, n_frames=NUM_FRAMES,
                                 status="skip_exists"))
            continue

        try:
            n = extract_clip(row.src_path, out_dir, NUM_FRAMES)
            status = "ok" if n >= NUM_FRAMES * 0.8 else "partial"
        except Exception as e:
            n, status = 0, f"error: {e}"

        results.append(dict(video_id=row.video_id, split=row.split, label=row.label,
                             method=row.method, out_dir=out_dir, n_frames=n, status=status))

        if i % 100 == 0 and i > 0:
            print_disk_usage()
            gc.collect()

    return pd.DataFrame(results)


if os.path.exists(CHECKPOINT_CSV):
    print("Tim thay checkpoint cu, se tiep tuc dua tren cac thu muc da co san.")

final_manifest = process_all(manifest, OUT_DIR)
final_manifest.to_csv(CHECKPOINT_CSV, index=False)

print("\n================ HOAN TAT TIEN XU LY CELEB-DF V2 (100 frames) ================")
print(final_manifest["status"].value_counts())
print_disk_usage()

## 5. Kiem tra ket qua -- so lieu that, anh mau

In [ ]:
print("=== So video xu ly THANH CONG theo label (0=real, 1=fake) ===")
ok_mask = final_manifest["status"].isin(["ok", "skip_exists"])
print(final_manifest[ok_mask].groupby(["label", "method"])["video_id"].count())

n_fail = (~ok_mask).sum()
print(f"\nSo video LOI/THIEU khung hinh: {n_fail} / {len(final_manifest)}")
if n_fail > 0:
    print(final_manifest[~ok_mask][["video_id", "status"]].head(20).to_string(index=False))

import matplotlib.pyplot as plt
sample_dirs = final_manifest[ok_mask].sample(min(6, ok_mask.sum()), random_state=42)
fig, axes = plt.subplots(1, len(sample_dirs), figsize=(3 * len(sample_dirs), 3))
if len(sample_dirs) == 1: axes = [axes]
for ax, (_, row) in zip(axes, sample_dirs.iterrows()):
    imgs = sorted(glob.glob(os.path.join(row["out_dir"], "*.jpg")))
    if imgs:
        img = cv2.cvtColor(cv2.imread(imgs[len(imgs)//2]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    ax.set_title(f"{row['label']}/{row['method']}\n{row['video_id'][:20]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_crops_preview.png"), dpi=100)
plt.show()

total, used, free = shutil.disk_usage(OUT_DIR)
print(f"\nTong dung luong cache: {used/1e9:.2f} GB")

## 6. Luu cache vinh vien

1. **Save Version -> Save & Run All (Commit)**.
2. Sau khi chay xong, vao tab **Output** -> **New Dataset** tu thu muc `/kaggle/working/celebdf_v2_rppg_cache/`.
3. Dat ten dataset: `celebdf-v2-rppg-cache`.
4. Dung dataset nay lam input cho **2 notebook eval-only** (`eval-celebdf-rppg.ipynb` va `eval-celebdf-can.ipynb`).

**Luu y:** Cache nay co **100 frames/video** (thay vi 32 frames cua cache Nhom A).
Nhanh rPPG can nhieu temporal context, nen 100 frames dam bao CLIP_LEN_EVAL=96 khong bi padding qua nhieu.